# Prophet ↔ single-cell dataset overlap

**Goal of `feat/train_pheno`:** fold phenotype-level (bulk) readouts from the **Prophet** datasets
into CellFlow as conditions that regulate single-cell generation. For that to be usable, the
`(cell_line, drug)` pairs we model in single-cell (sciplex / tahoe in `unipert/`) must have a
corresponding phenotype in a Prophet dataset.

This notebook quantifies, with exact set operations:
1. **Inventory** — per dataset: #cell lines, #perturbations, readout type, size.
2. **Cell-line & drug overlap** across all datasets (Jaccard + shared counts).
3. **Single-cell → Prophet coverage** — the decisive number: what fraction of each single-cell
   dataset's drugs / `(cell_line, drug)` pairs is reachable in each Prophet dataset.

Reads `.h5ad` via `h5py` (no anndata dep — the env has a zarr/anndata clash) and CSVs in chunks
(LINCS is ~8 GB). The script form lives in `analysis/prophet_overlap.py`.


In [ ]:
import re, pickle
from pathlib import Path
from collections import Counter

import h5py, numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

DATA    = Path("/lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow")
PROPHET = DATA / "prophet"
UNIPERT = DATA / "unipert"
TAB = Path("tables");  TAB.mkdir(exist_ok=True)
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
CACHE = Path("_cache"); CACHE.mkdir(exist_ok=True)

CONTROLS = {"control","dmso","none","nan","","non-targeting","nontargeting","neg","vehicle"}

## Identifier normalization

Cell-line names are inconsistent across datasets (`'BT-20 cell'`, `'AU565 cell'`, `'K-562'`), and
**tahoe single-cell files store Cellosaurus accessions** (`'CVCL_1239'`) instead of names — so we
recover the tahoe cell name from the filename. Drug names are normalized loosely (lowercase, strip
parentheticals/punctuation). This is a *first-pass* harmonization; production use should map to
canonical ids (Cellosaurus for cell lines, SMILES/InChIKey for drugs).

In [ ]:
def norm_cell(s):
    s = str(s).upper().strip()
    s = re.sub(r"\s*CELL LINE\s*$", "", s)
    s = re.sub(r"\s+CELL\s*$", "", s)
    return re.sub(r"[^A-Z0-9]", "", s)

def norm_drug(s):
    s = str(s).lower().strip()
    s = re.sub(r"\(.*?\)", "", s)          # drop parenthetical aliases
    return re.sub(r"[^a-z0-9]", "", s)

def split_perts(label):
    out = []
    for p in re.split(r"[;_+]| and ", str(label)):
        n = norm_drug(p)
        if n and p.strip().lower() not in CONTROLS:
            out.append(n)
    return out

In [ ]:
def _read_cat(node):
    if isinstance(node, h5py.Group):                  # categorical column
        cats = np.array([c.decode() if isinstance(c, bytes) else c for c in node["categories"][:]])
        return cats, node["codes"][:]
    vals = np.array([v.decode() if isinstance(v, bytes) else v for v in node[:]])
    return np.unique(vals, return_inverse=True)

def load_h5ad(path, cell_col, drug_col, cell_override=None):
    with h5py.File(path, "r") as f:
        obs = f["obs"]
        cl_cats, cl_codes = _read_cat(obs[cell_col])
        dr_cats, dr_codes = _read_cat(obs[drug_col])
        n = len(cl_codes)
        try:    n_readout = f["var"]["_index"].shape[0]
        except Exception: n_readout = None
    if cell_override is not None:                      # opaque id (Cellosaurus) -> pin name
        one = norm_cell(cell_override); cl_norm = np.array([one]*len(cl_cats)); cell_lines = {one}
    else:
        cl_norm = np.array([norm_cell(c) for c in cl_cats]); cell_lines = {c for c in cl_norm if c}
    drugs = set()
    for d in dr_cats: drugs |= set(split_perts(d))
    dr_full = np.array([norm_drug(d) for d in dr_cats])
    m = (cl_codes >= 0) & (dr_codes >= 0)
    pairs = set(zip(cl_norm[cl_codes[m]].tolist(), dr_full[dr_codes[m]].tolist()))
    pairs = {(c, d) for c, d in pairs if c and d and d not in CONTROLS}
    return dict(cell_lines=cell_lines, drugs=drugs, pairs=pairs, n_rows=int(n), n_readout=n_readout)

def load_csv(path, chunksize=1_000_000):
    keep = {"cell_line","iv1","iv2","phenotype"}
    cell_lines, drugs, pairs, phenos, n = set(), set(), set(), set(), 0
    for chunk in pd.read_csv(path, chunksize=chunksize, low_memory=False, usecols=lambda c: c in keep):
        n += len(chunk)
        if "cell_line" in chunk: cell_lines |= {norm_cell(c) for c in chunk["cell_line"].dropna().unique()}
        for col in ("iv1","iv2"):
            if col in chunk:
                for v in chunk[col].dropna().unique(): drugs |= set(split_perts(v))
        if "iv1" in chunk:
            pp = chunk[["cell_line","iv1"]].dropna().drop_duplicates()
            for c, d in zip(pp["cell_line"], pp["iv1"]):
                nc, nd = norm_cell(c), norm_drug(d)
                if nc and nd and nd not in CONTROLS: pairs.add((nc, nd))
        if "phenotype" in chunk: phenos |= set(chunk["phenotype"].dropna().astype(str).unique())
    cell_lines.discard("")
    return dict(cell_lines=cell_lines, drugs=drugs, pairs=pairs, n_rows=int(n), n_readout=len(phenos))

In [ ]:
PROPHET_H5AD = [
    ("GDSC",  "pert_compound", "drug singleton",   "ln(IC50) / viability"),
    ("GDSC2", "pert_compound", "drug combination", "ln(IC50) / viability"),
    ("CTRP",  "pert_compound", "drug singleton",   "IC50 (CellTiter-Glo)"),
    ("PRISM", "pert_compound", "drug singleton",   "Log2FC / MFI"),
    ("SCORE", "pert_name",     "CRISPR singleton", "essentiality (CERES)"),
]
PROPHET_CSV = [
    ("Horlbeck", "Horlbeck_dataset.csv", "CRISPR combination", "viability"),
    ("JUMP",     "JUMP_dataset.csv",     "drug/CRISPR",        "Cell Painting"),
    ("JUMPcr",   "JUMPcr_dataset.csv",   "CRISPR",             "Cell Painting"),
    ("JUMPsm",   "JUMPsm_dataset.csv",   "drug",               "Cell Painting"),
    ("LINCS",    "LINCS_dataset.csv",    "drug singleton",     "RNA (fluorescence)"),
    ("Shifrut",  "Shifrut_dataset.csv",  "CRISPR",             "proliferation"),
    ("PRISM_csv","PRISM_dataset.csv",    "drug singleton",     "Log2FC"),
]

def tahoe_cellname(stem):
    return stem[len("tahoe_"):].replace("_", "") if stem.startswith("tahoe_") else None

In [ ]:
def build_inventory(use_cache=True):
    rows, sets = [], {}
    pcache = CACHE / "prophet_sets.pkl"
    if use_cache and pcache.exists():
        prophet_sets, prophet_rows = pickle.load(open(pcache, "rb"))
        print("[cache] loaded Prophet sets")
    else:
        prophet_sets, prophet_rows = {}, []
        for name, col, pert, ro in PROPHET_H5AD:
            print("loading", name); d = load_h5ad(PROPHET/f"{name}.h5ad", "cell_line", col)
            prophet_sets[name] = d
            prophet_rows.append(dict(dataset=name, source="prophet", perturbation=pert, readout=ro,
                n_cell_lines=len(d["cell_lines"]), n_drugs=len(d["drugs"]), n_pairs=len(d["pairs"]),
                n_rows=d["n_rows"], n_readout_dims=d["n_readout"]))
        for name, fname, pert, ro in PROPHET_CSV:
            print("loading", name); d = load_csv(PROPHET/fname)
            prophet_sets[name] = d
            prophet_rows.append(dict(dataset=name, source="prophet", perturbation=pert, readout=ro,
                n_cell_lines=len(d["cell_lines"]), n_drugs=len(d["drugs"]), n_pairs=len(d["pairs"]),
                n_rows=d["n_rows"], n_readout_dims=d["n_readout"]))
        pickle.dump((prophet_sets, prophet_rows), open(pcache, "wb"))
    sets.update(prophet_sets); rows.extend(prophet_rows)

    for path in sorted(UNIPERT.glob("*_w_emb.h5ad")):
        stem = path.stem.replace("_w_emb", ""); name = "sc:" + stem
        ov = tahoe_cellname(stem)
        d = load_h5ad(path, "cell_line", "drug_0", cell_override=ov)
        sets[name] = d
        rows.append(dict(dataset=name, source="single-cell", perturbation="drug (sc)", readout="scRNA",
            n_cell_lines=len(d["cell_lines"]), n_drugs=len(d["drugs"]), n_pairs=len(d["pairs"]),
            n_rows=d["n_rows"], n_readout_dims=d["n_readout"]))
    return pd.DataFrame(rows), sets

# NB: first run reads the ~8GB LINCS csv (minutes); afterwards Prophet sets are cached.
inv, sets = build_inventory()
inv.sort_values(["source", "dataset"]).reset_index(drop=True)

## Overlap matrices

In [ ]:
def jaccard(a, b): return len(a & b) / len(a | b) if a and b else 0.0

def matrix(names, key, fn=jaccard):
    return pd.DataFrame([[fn(sets[a][key], sets[b][key]) for b in names] for a in names],
                        index=names, columns=names)

def heat(df, title, fmt="{:.2f}", cmap="viridis"):
    fig, ax = plt.subplots(figsize=(0.55*len(df.columns)+4, 0.5*len(df.index)+3))
    im = ax.imshow(df.values, cmap=cmap, aspect="auto")
    ax.set_xticks(range(len(df.columns))); ax.set_xticklabels(df.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(len(df.index)));   ax.set_yticklabels(df.index, fontsize=7)
    for i in range(len(df.index)):
        for j in range(len(df.columns)):
            v = df.values[i, j]
            ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=6,
                    color="white" if im.norm(v) < 0.6 else "black")
    ax.set_title(title); fig.colorbar(im, ax=ax, shrink=0.7); fig.tight_layout(); plt.show()

names      = list(inv["dataset"])
drug_names = [n for n in names if sets[n]["drugs"]]
heat(matrix(names, "cell_lines"),      "Cell-line overlap (Jaccard)")
heat(matrix(drug_names, "drugs"),      "Drug overlap (Jaccard)")

## Single-cell → Prophet coverage (the decisive table)

In [ ]:
sc_names      = [n for n in names if n.startswith("sc:")]
prophet_names = [n for n in names if not n.startswith("sc:")]

rows = []
for sc in sc_names:
    S = sets[sc]
    for pn in prophet_names:
        P = sets[pn]
        rows.append(dict(single_cell=sc, prophet=pn,
            cl_covered=len(S["cell_lines"] & P["cell_lines"]),
            drugs=len(S["drugs"]), drug_covered=len(S["drugs"] & P["drugs"]),
            drug_frac=round(len(S["drugs"] & P["drugs"])/max(len(S["drugs"]),1), 3),
            pairs=len(S["pairs"]), pair_covered=len(S["pairs"] & P["pairs"]),
            pair_frac=round(len(S["pairs"] & P["pairs"])/max(len(S["pairs"]),1), 3)))
cov = pd.DataFrame(rows)
cov.to_csv(TAB/"singlecell_to_prophet_coverage.csv", index=False)

# heatmap of pair-coverage fraction (single-cell x prophet)
piv = cov.pivot(index="single_cell", columns="prophet", values="pair_frac").fillna(0)
heat(piv, "(cell_line, drug) pair coverage of single-cell by Prophet", cmap="magma")
cov[cov.pair_frac > 0].sort_values("pair_frac", ascending=False).head(20)

## Findings

**Inventory (what each Prophet dataset offers).** Drug-response scalars come with broad cell-line
panels — **PRISM** (567 cell lines × 4,028 drugs, Log2FC), **GDSC** (967 × 298, ln IC50),
**CTRP** (887 × 696, IC50) — while the richer phenotypes are panel-limited: **LINCS** (RNA, 978-dim)
has only **69** cell lines and **JUMP** (Cell Painting, 200-dim) only **2**. **SCORE/Horlbeck/JUMPcr**
are CRISPR (genetic), orthogonal to the drug pilot. A long tail of cell lines is dataset-unique
(471 appear in only one dataset), with a shared core (one cell line appears in 14 datasets).

**Coverage of the pilots (the decisive result).**

| single-cell | best Prophet | drug coverage | (cell_line, drug) pair coverage |
|---|---|---|---|
| sciplex2 | PRISM | 3/3 (1.00) | **3/3 (1.00)** |
| sciplex3 | PRISM | 126/180 (0.70) | **252/540 (0.47)**; GDSC 94 (0.17) |
| combosciplex | PRISM | 6/7 (0.86) | **6/7 (0.86)** |
| tahoe (A549, ASPC1, H4, PANC1) | PRISM | 220/347 (0.63) | **220 pairs (0.63)**; GDSC ~35 |
| tahoe (HOP62, SNU1) | GDSC | — | not in PRISM panel; ~35 via GDSC |

So **drug-response phenotype (PRISM Log2FC, GDSC IC50) is directly reachable** for roughly half to
all of the exact `(cell_line, drug)` pairs in the pilots — phenotype-as-condition is testable *now*,
no new data needed.

**What is NOT reachable.** Cell Painting (JUMP, 2 cell lines) and transcriptional (LINCS, 69) phenotypes
do **not** cover the pilots' cell lines → ~0 pair coverage. The bottleneck for those modalities is the
**cell-line panel**, not the drugs.

**Harmonization is the real engineering cost.** This already bit us: tahoe stores **Cellosaurus ids**,
fixed here via filename, but at scale you need a canonical cell-line map (Cellosaurus ↔ name ↔ DepMap)
and drug name ↔ SMILES/InChIKey matching. The fuzzy `norm_drug`/`norm_cell` here is a floor on true
overlap, not the ceiling.


## Implications & next steps

1. **Test phenotype conditioning on the overlap.** Add the PRISM (and/or GDSC) drug-response scalar
   as a new condition key for the sciplex/tahoe `(cell_line, drug)` pairs that overlap (~0.47–1.0).
   The set-encoder picks up new condition keys automatically (just a new `rep_key`), and tolerates
   the pairs where the phenotype is missing. This is the cleaner test of *"does phenotype add signal"*
   than the prophet **drug-embedding** (which did not beat random).
2. **Pick the phenotype modality by coverage.** For the current pilots, only the **scalar drug-response**
   datasets are usable. Cell Painting / LINCS-RNA need a single-cell panel that intersects their cell
   lines (or new data) before they can help.
3. **Invest in identifier harmonization** (Cellosaurus + drug-synonym/SMILES tables) before scaling to
   many datasets — it directly determines how much phenotype supervision is reachable.
